In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDs


In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them.

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
#### 1. THE MOCK DATABASE (Stateful)

In [5]:
db = {
    "mark_tan": {"annual": 14, "medical": 14, "family": 3},
    "jane_doe": {"annual": 2, "medical": 10, "family": 0}
}

#### defining our tools

##### Tool for retrieving our leave balance

In [6]:
def get_leave_balance(employee_id, leave_type):
    print(f"\n🔎 [DATABASE READ] Querying {leave_type} balance for {employee_id}...") 
    
    user_data = db.get(employee_id)
    if not user_data:
        return json.dumps({"error": "Employee not found"})
    
    balance = user_data.get(leave_type.lower())
    if balance is None:
        return json.dumps({"error": f"Invalid leave type: {leave_type}"})
    
    print(f"   ↳ Result: {balance} days found.")
    return json.dumps({"balance": balance, "unit": "days"})

##### Tool for retrieving for submitting our leave request

In [7]:
def submit_leave_request(employee_id, leave_type, days):
    print(f"\n📝 [DATABASE WRITE] Attempting to deduct {days} days of {leave_type} leave for {employee_id}...")
    
    user_data = db.get(employee_id)
    if not user_data:
        return json.dumps({"error": "Employee not found"})

    current_balance = user_data.get(leave_type.lower(), 0)
    
    # --- LOGIC: CHECK BALANCE ---
    if current_balance < days:
        print(f"   ❌ Failed: Insufficient balance ({current_balance} < {days})")
        return json.dumps({"status": "failed", "message": f"Insufficient balance. You only have {current_balance} days."})
    
    # --- LOGIC: DEDUCT BALANCE ---
    db[employee_id][leave_type.lower()] -= days
    new_balance = db[employee_id][leave_type.lower()]
    
    print(f"   ✅ Success: Balance updated. Old: {current_balance} -> New: {new_balance}")
    return json.dumps({"status": "success", "message": f"Approved. Remaining balance: {new_balance} days."})

##### creating dictionary mapping to map our functions

In [8]:
available_functions = {
    "get_leave_balance": get_leave_balance,
    "submit_leave_request": submit_leave_request,
}

##### Configuring our agent

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_leave_balance",
            "description": "Get the remaining leave days for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "leave_type": {"type": "string", "enum": ["annual", "medical", "family"]},
                },
                "required": ["employee_id", "leave_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_leave_request",
            "description": "Submit a leave application. DEDUCTS from balance.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "leave_type": {"type": "string", "enum": ["annual", "medical", "family"]},
                    "days": {"type": "integer"},
                },
                "required": ["employee_id", "leave_type", "days"],
            },
        },
    }
]

##### creating our agent loop

In [10]:
def agent_chat(message, history):
    print("\n" + "="*50)
    print(f"📨 NEW USER MESSAGE: {message}")
    print("="*50)

    # 1. Prepare History
    history_formatted = [{"role": h["role"], "content": h["content"]} for h in history]
    system_prompt = "You are a generic HR assistant. Current Employee ID: mark_tan."
    messages = [{"role": "system", "content": system_prompt}] + history_formatted + [{"role": "user", "content": message}]

    # 2. First Pass: Thinking
    print("🤖 [AGENT] Thinking... (Analyzing request vs Tools)")
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto", 
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # 3. Check if Tool is needed
    if tool_calls:
        print(f"💡 [AGENT] Decision: I need to use {len(tool_calls)} tool(s).")
        messages.append(response_message) 

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"   👉 Invoking Tool: {function_name}")
            print(f"   📦 Arguments: {function_args}")
            
            # EXECUTE FUNCTION
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            
            print(f"   🔙 Tool Output: {function_response}")
            
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
            
        # 4. Second Pass: Final Answer
        print("🤖 [AGENT] Reading tool outputs and formulating answer...")
        second_response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        final_answer = second_response.choices[0].message.content
        print(f"💬 [AGENT] Final Reply: {final_answer}\n")
        return final_answer

    else:
        print("🤷 [AGENT] No tools needed. Replying directly.")
        return response_message.content

##### our gradio GUI

In [11]:
view = gr.ChatInterface(
    fn=agent_chat,
    type="messages",
    title="SimpliAsk HR Agent", 
    description="Check your terminal to see the Agent Traces running in real-time.",
    examples=["I have 2 days of annual leave left?", "Apply for 5 days of annual leave."],
)

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://22cc37ab9bcd644013.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



📨 NEW USER MESSAGE: what is my leave balance
🤖 [AGENT] Thinking... (Analyzing request vs Tools)
💡 [AGENT] Decision: I need to use 3 tool(s).
   👉 Invoking Tool: get_leave_balance
   📦 Arguments: {'employee_id': 'mark_tan', 'leave_type': 'annual'}

🔎 [DATABASE READ] Querying annual balance for mark_tan...
   ↳ Result: 14 days found.
   🔙 Tool Output: {"balance": 14, "unit": "days"}
   👉 Invoking Tool: get_leave_balance
   📦 Arguments: {'employee_id': 'mark_tan', 'leave_type': 'medical'}

🔎 [DATABASE READ] Querying medical balance for mark_tan...
   ↳ Result: 14 days found.
   🔙 Tool Output: {"balance": 14, "unit": "days"}
   👉 Invoking Tool: get_leave_balance
   📦 Arguments: {'employee_id': 'mark_tan', 'leave_type': 'family'}

🔎 [DATABASE READ] Querying family balance for mark_tan...
   ↳ Result: 3 days found.
   🔙 Tool Output: {"balance": 3, "unit": "days"}
🤖 [AGENT] Reading tool outputs and formulating answer...
💬 [AGENT] Final Reply: Your current leave balances are as follows:

- **